# Ztracker square open field: a simpler analysis

Only two analysis functions are used:

- `analyze_single_OF(path, ...)` loads one session and computes its four-quarter summary.
- `analyze_all_OF(FISH_DATA, ...)` loops over fish and pre / LED / post files,
  adding the fish ID, genotype and session directly to the result.

There are no genotype wrappers or input-type checks. The analysis assumes the
regular saved schema. The existing fish IDs, file paths and supplied genotype
values are retained below; fill any genotype still set to `None`.
Quarter 1 remains the target: **x < 0.5, y < 0.5**.

Occupancy uses elapsed VR time and excludes calibration, long gaps and invalid
position endpoints. Bout measurements retain the previous deduplication, quality
mask, short-quiet-gap merging and censoring rules. Counts refer to bout onsets;
rates are bouts per observable minute in that quarter. Durations use complete
bouts. The parameter values are analysis choices, not newly fitted thresholds.

Run from top to bottom. Dependencies: numpy, h5py, pandas, matplotlib, seaborn.
No Ztracker or zfish installation is needed. The separate `analyze_GDG.py` script
handles the GDG task using the recorded `bout_onset` events and its fixed 20/5 s
period denominators.


In [2]:
"""Straightforward square open-field analysis: one session, then all fish.

No input-type validation or genotype wrappers. Data use the regular Ztracker
schema. Occupancy is VR dwell time; bouts use real, deduplicated saved active
observations. Quality masks and censoring are measurement rules, not type checks.
"""

import json
from pathlib import Path

import h5py
import numpy as np
from tqdm import tqdm


def analyze_single_OF(
    path, 
    time_window=None, 
    max_gap_s=0.5,
    max_observation_gap_s=0.15, 
    merge_gap_s=0.1,
):
    """Return four rows of occupancy and bout properties for one session.

    Quarter 1: x<0.5,y<0.5; 2: x>=0.5,y<0.5; 3: x<0.5,y>=0.5; 4: the rest.
    time_window, when supplied, is (start, stop) in active assay seconds.
    """
    path = Path(path)
    if path.is_dir():
        path = path / "res.h5"

    # 1. Load the controller state and distinct, usable motion observations.
    with h5py.File(path, "r") as file:
        state = file["state"][:]
        recording_status = file.attrs["status"]
        good = (
            state["tracking_fresh"] & state["motion_valid"]
            & ~state["motion_held"] & ~state["motion_filled"]
            & ~state["motion_suppressed"]
        )
        candidates = np.flatnonzero(good & state["decoded_this_update"])
        seen, motion_rows, active = set(), [], []
        for start in range(0, len(state), 1024):
            selected = candidates[(candidates >= start) & (candidates < start + 1024)]
            if not len(selected):
                continue
            snapshots = file["states"].asstr()[start:start + 1024]
            for row in selected:
                identity = (
                    state["source_stream_id"][row], state["motion_observed_s"][row]
                )
                if identity in seen:
                    continue
                seen.add(identity)
                if not (
                    state["host_time_s"][0] <= identity[1]
                    <= state["host_time_s"][row]
                ):
                    good[row] = False
                    continue
                snapshot = json.loads(snapshots[row - start])
                active.append(snapshot["system_state"]["motion"]["active"])
                motion_rows.append(row)

    # 2. Calculate elapsed VR time, omitting calibration, long gaps and invalid
    # position endpoints. Clip a requested window without rounding whole rows.
    time = state["host_time_s"]
    dt = np.diff(time)
    assay_time = state["assay_time_s"]
    da = np.diff(assay_time)
    left, right = time[:-1].copy(), time[1:].copy()
    if time_window is not None:
        for boundary, target in zip(time_window, (left, right)):
            fraction = np.divide(
                boundary - assay_time[:-1], da,
                out=np.zeros_like(da), where=da > 0,
            )
            target[:] = time[:-1] + np.clip(fraction, 0, 1) * dt
    x, y = state["xpos"], state["ypos"]
    position_ok = np.isfinite(x) & np.isfinite(y)
    position_ok &= (x >= 0) & (x <= 1) & (y >= 0) & (y <= 1)
    valid = (da > 0) & (dt > 0) & (dt <= max_gap_s)
    valid &= position_ok[:-1] & position_ok[1:]
    weights = np.where(valid, np.maximum(0, right - left), 0)
    quarter = 1 + (x >= 0.5).astype(int) + 2 * (y >= 0.5).astype(int)
    dwell = np.array([weights[quarter[:-1] == q].sum() for q in range(1, 5)])

    # 3. Find continuous brackets between real motion observations. These
    # provide the denominator for measured-bout rates and active fractions.
    motion_rows = np.asarray(motion_rows, dtype=int)
    active = np.asarray(active, dtype=bool)
    motion_time = state["motion_observed_s"][motion_rows]
    connected = np.zeros(max(0, len(motion_rows) - 1), dtype=bool)
    bad_prefix = np.r_[0, np.cumsum(~good)]
    exposure, active_time = np.zeros(4), np.zeros(4)
    for index in range(len(motion_time) - 1):
        start, stop = motion_time[index:index + 2]
        first_row, next_row = motion_rows[index:index + 2]
        if not 0 < stop - start <= max_observation_gap_s + 1e-12:
            continue
        if state["source_stream_id"][first_row] != state["source_stream_id"][next_row]:
            continue
        if bad_prefix[next_row + 1] != bad_prefix[first_row]:
            continue
        begin = max(0, np.searchsorted(time, start, side="right") - 1)
        end = min(len(weights), np.searchsorted(time, stop, side="left"))
        overlap = np.maximum(
            0, np.minimum(right[begin:end], stop) - np.maximum(left[begin:end], start)
        )
        overlap[weights[begin:end] == 0] = 0
        if not np.isclose(overlap.sum(), stop - start, atol=1e-8, rtol=1e-7):
            continue
        connected[index] = True
        for q in range(1, 5):
            duration = overlap[quarter[begin:end] == q].sum()
            exposure[q - 1] += duration
            active_time[q - 1] += duration * active[index]

    # 4. Count onsets once in their onset quarter. Merge only short observed
    # quiet gaps. An already-active segment start is left-censored, not a new
    # onset; an unconfirmed end cannot supply a complete-bout duration.
    counts, complete = np.zeros(4, dtype=int), np.zeros(4, dtype=int)
    left_censored, right_censored = np.zeros(4, dtype=int), np.zeros(4, dtype=int)
    durations = [[] for _ in range(4)]
    boundaries = np.r_[0, np.flatnonzero(~connected) + 1, len(motion_time)]
    for begin, end in zip(boundaries[:-1], boundaries[1:]):
        if end - begin < 2:
            continue
        t, a = motion_time[begin:end], active[begin:end]
        starts = np.flatnonzero(a & ~np.r_[False, a[:-1]])
        stops = np.flatnonzero(a & ~np.r_[a[1:], False]) + 1
        runs = []
        for onset, offset in zip(starts, stops):
            if runs and t[onset] - t[runs[-1][1]] < merge_gap_s - 1e-12:
                runs[-1][1] = offset
            else:
                runs.append([onset, offset])
        for number, (onset, offset) in enumerate(runs):
            q = quarter[np.searchsorted(time, t[onset], side="right") - 1] - 1
            missing_start = onset == 0
            missing_end = offset == len(t)
            if not missing_end and number == len(runs) - 1:
                missing_end = t[-1] - t[offset] < merge_gap_s - 1e-12
            left_censored[q] += missing_start
            right_censored[q] += missing_end
            if not missing_start:
                counts[q] += 1
                if not missing_end:
                    complete[q] += 1
                    durations[q].append(t[offset] - t[onset])

    # 5. Return four equally sized summary rows.
    result = {
        "Quarter": [], "Occupancy": [], "Dwell time (s)": [],
        "Swim bout count": [], "Swim bout rate (per min)": [],
        "Bout-observable time (s)": [], "Swim-active time (s)": [],
        "Swim fraction": [], "Mean bout duration (s)": [],
        "Complete bout count": [], "Left-censored bout fragments": [],
        "Right-censored bout fragments": [], "Bout observation coverage": [],
        "Analyzed time (s)": [], "Recording status": [],
    }
    for q in range(4):
        result["Quarter"].append(q + 1)
        result["Occupancy"].append(dwell[q] / dwell.sum() if dwell.sum() else np.nan)
        result["Dwell time (s)"].append(dwell[q])
        result["Swim bout count"].append(counts[q] if exposure.sum() else np.nan)
        result["Swim bout rate (per min)"].append(
            counts[q] * 60 / exposure[q] if exposure[q] else np.nan
        )
        result["Bout-observable time (s)"].append(exposure[q])
        result["Swim-active time (s)"].append(active_time[q])
        result["Swim fraction"].append(
            active_time[q] / exposure[q] if exposure[q] else np.nan
        )
        result["Mean bout duration (s)"].append(
            np.mean(durations[q]) if durations[q] else np.nan
        )
        result["Complete bout count"].append(complete[q] if exposure.sum() else np.nan)
        result["Left-censored bout fragments"].append(left_censored[q])
        result["Right-censored bout fragments"].append(right_censored[q])
        result["Bout observation coverage"].append(
            exposure[q] / dwell[q] if dwell[q] else np.nan
        )
        result["Analyzed time (s)"].append(dwell.sum())
        result["Recording status"].append(recording_status)
    return result


def analyze_all_OF(all_path, **options):
    """Analyze {fish_id: {'genotype': label, 'files': [pre, LED, post]}}."""
    result = {"Fish ID": [], "Genotype": [], "Session": []}
    for fish_id, info in tqdm(all_path.items()):
        for session, path in zip(("pre", "LED", "post"), info["files"]):
            single = analyze_single_OF(path, **options)
            result["Fish ID"].extend([fish_id] * 4)
            result["Genotype"].extend([info["genotype"]] * 4)
            result["Session"].extend([session] * 4)
            for name, values in single.items():
                result.setdefault(name, []).extend(values)
    return result


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

GENOTYPE_ORDER = ["positive", "negative"]
GENOTYPE_PALETTE = {"positive": "#275D8C", "negative": "#C76850"}
SESSION_ORDER = ["pre", "LED", "post"]

# Enter each fish's genotype; files are ordered pre, LED, post.
FISH_DATA = {
        10290: {'genotype': "Positive",
                'files': [r'E:\Open Field\Square OF\Dat-CoChr Positive\10290\S3-OF-Pre\res.h5',
                          r'E:\Open Field\Square OF\Dat-CoChr Positive\10290\S4-OF-LED\res.h5',
                          r'E:\Open Field\Square OF\Dat-CoChr Positive\10290\S5-OF-Post\res.h5']},
 10292: {'genotype': "Positive",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Positive\10292\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10292\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10292\S5-OF-Post\res.h5']},
 10293: {'genotype': "Positive",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Positive\10293\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10293\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10293\S5-OF-Post\res.h5']},
 10294: {'genotype': "Positive",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Positive\10294\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10294\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10294\S5-OF-Post\res.h5']},
 10295: {'genotype': "Positive",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Positive\10295\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10295\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10295\S5-OF-Post\res.h5']},
 10296: {'genotype': "Positive",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Positive\10296\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10296\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Positive\10296\S5-OF-Post\res.h5']},
 10297: {'genotype': "Negative",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Negative\10297\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10297\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10297\S5-OF-Post\res.h5']},
 10298: {'genotype': "Negative",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Negative\10298\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10298\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10298\S5-OF-Post\res.h5']},
 10299: {'genotype': "Negative",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Negative\10299\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10299\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10299\S5-OF-Post\res.h5']},
 10300: {'genotype': "Negative",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Negative\10300\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10300\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10300\S5-OF-Post\res.h5']},
 10301: {'genotype': "Negative",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Negative\10301\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10301\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10301\S5-OF-Post\res.h5']},
 10302: {'genotype': "Negative",
         'files': [r'E:\Open Field\Square OF\Dat-CoChr Negative\10302\S3-OF-Pre\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10302\S4-OF-LED\res.h5',
                   r'E:\Open Field\Square OF\Dat-CoChr Negative\10302\S5-OF-Post\res.h5']}
}
CONFIG = {
    "time_window": None,
    "max_gap_s": 0.5,
    "max_observation_gap_s": 0.15,
    "merge_gap_s": 0.1,
}


In [ ]:
result = analyze_all_OF(FISH_DATA, **CONFIG)
table = pd.DataFrame(result)
table["Session"] = pd.Categorical(
    table["Session"], categories=SESSION_ORDER, ordered=True
)
display(table)


`Bout-observable time` can be shorter than VR dwell time because held/filled
motion is excluded from measured-bout analysis. Compare observation coverage
and censored fragments across sessions before interpreting the bout metrics.


In [ ]:
# Select the metrics/quarters you want to display.
metrics = [
    "Occupancy", "Swim bout count", "Swim bout rate (per min)",
    "Swim fraction", "Mean bout duration (s)",
]
for quarter in [1, 2, 3, 4]:
    subdata = table.loc[table["Quarter"] == quarter].sort_values(
        ["Fish ID", "Session"]
    )
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(4, 3))
        sns.lineplot(
            data=subdata, x="Session", y=metric, hue="Genotype",
            hue_order=GENOTYPE_ORDER, palette=GENOTYPE_PALETTE,
            units="Fish ID", estimator=None,
            linewidth=0.6, alpha=0.35, legend=False, ax=ax,
        )
        sns.lineplot(
            data=subdata, x="Session", y=metric, hue="Genotype",
            hue_order=GENOTYPE_ORDER, palette=GENOTYPE_PALETTE,
            estimator="mean", errorbar="se", marker="s", markersize=5,
            markeredgecolor="none", linewidth=2, err_style="bars",
            err_kws={"capsize": 3, "linewidth": 2, "capthick": 2}, ax=ax,
        )
        sns.stripplot(
            data=subdata, x="Session", y=metric, hue="Genotype",
            order=SESSION_ORDER, hue_order=GENOTYPE_ORDER,
            palette=GENOTYPE_PALETTE, dodge=False, jitter=0,
            size=5, alpha=0.7, linewidth=0, legend=False, ax=ax,
        )
        if metric == "Occupancy":
            ax.axhline(0.25, color="0.4", ls="--", lw=0.7)
            ax.set_ylim(0, 1)
        else:
            ax.set_ylim(bottom=0)
        ax.set_title(f"Quarter {quarter}")
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(title="Genotype", frameon=False)
        fig.tight_layout()
        plt.show()


In [ ]:
# Changes remain paired by fish, genotype and quarter.
changes = {}
for metric in metrics:
    paired = table.pivot(
        index=["Fish ID", "Genotype", "Quarter"],
        columns="Session", values=metric,
    ).reindex(columns=SESSION_ORDER)
    changes[metric] = pd.DataFrame({
        "LED - pre": paired["LED"] - paired["pre"],
        "post - pre": paired["post"] - paired["pre"],
        "post - LED": paired["post"] - paired["LED"],
    })
    print(metric)